In [ ]:
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.append('..')
from environment.trading_env import TradingEnv
from experts.tqc_expert import TQCAgent

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
ASSET = "EURUSD"  # change to your asset
CONFIG_PATH = "../configs/experts.yaml"
TEST_DATA_PATH = f"../data/processed/{ASSET}/train.csv"
MODEL_PATH = f"../checkpoints/{ASSET}/no_diversity/TQC/tqc_expert.pt"

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)
tqc_cfg = cfg["tqc"]

In [ ]:
# Load test data
test_df = pd.read_csv(TEST_DATA_PATH, index_col=0, parse_dates=True)
print(f"Test data shape: {test_df.shape}")

# Load checkpoint
ckpt = torch.load(MODEL_PATH, map_location='cpu')
print(f"Checkpoint keys: {list(ckpt.keys())}")

# Get expected state dimension from actor network
# TQC actor has a `net` component; we can inspect its first layer
actor_state = ckpt["actor"]
# The actor's first layer is typically 'net.0.weight' or similar.
# We'll look for the first linear layer key.
first_layer_key = None
for key in actor_state.keys():
    if "weight" in key and ("net" in key or "0" in key):
        first_layer_key = key
        break
if first_layer_key is None:
    raise KeyError("Could not find first layer in actor state dict.")

expected_input_dim = actor_state[first_layer_key].shape[1]
expected_features = expected_input_dim - 6   # 6 environment state features
print(f"Checkpoint expects {expected_features} features (input dim {expected_input_dim}).")

# Use config features if they match
config_features = tqc_cfg.get("features", [])
if len(config_features) == expected_features:
    features = config_features
else:
    print(f"Config has {len(config_features)} features, but checkpoint expects {expected_features}.")
    # Option 1: adapt by taking first N columns from test data (excluding price/volume)
    exclude = ["open", "high", "low", "close", "volume", "log_volume", "returns", "return_5"]
    candidates = [c for c in test_df.columns if c not in exclude]
    if len(candidates) >= expected_features:
        features = candidates[:expected_features]
    else:
        features = list(test_df.columns[:expected_features])
    print("Using adapted features:", features)

In [ ]:
# Environment – continuous actions
asset_type = "crypto" if "USDT" in ASSET else "forex"
window_size = cfg.get("window_size", 20)
env_params = {
    "df": test_df,
    "feature_cols": features,
    "expert_id": 4,  # TQC = expert 4
    "asset_type": asset_type,
    "contract_size": 1.0 if asset_type == "crypto" else 100000.0,
    "commission_rate": 0.0005 if asset_type == "crypto" else 1.5/100000.0,
    "spread_rate": 0.0002 if asset_type == "crypto" else 0.00005,
    "slippage_rate": 0.0001 if asset_type == "crypto" else 0.00001,
    "leverage": 5 if asset_type == "crypto" else 30,
    "maintenance_margin_ratio": 0.5,
    "initial_balance": 10000.0,
    "window_size": window_size,
    "reward_scale": 1.0,
    "clip_reward": (-10.0, 10.0),
    "max_position_fraction": 0.95,
    "continuous_actions": True,   # TQC uses continuous actions
}
env = TradingEnv(**env_params)

# TQC Agent
state_dim = len(features) + 6
agent = TQCAgent(
    state_dim=state_dim,
    action_dim=1,                  # continuous, single action
    lr=tqc_cfg.get("learning_rate", 3e-4),
    lr_actor=tqc_cfg.get("lr_actor"),
    lr_critic=tqc_cfg.get("lr_critic"),
    gamma=tqc_cfg.get("gamma", 0.99),
    tau=tqc_cfg.get("tau", 0.005),
    buffer_size=tqc_cfg.get("buffer_size", 100000),
    batch_size=tqc_cfg.get("batch_size", 256),
    warmup_steps=tqc_cfg.get("warmup_steps", 1000),
    hidden_dim=tqc_cfg.get("hidden_dim", 256),
    actor_hidden_dim=tqc_cfg.get("actor_hidden_dim"),
    critic_hidden_dim=tqc_cfg.get("critic_hidden_dim"),
    actor_num_layers=tqc_cfg.get("actor_num_layers", 2),
    critic_num_layers=tqc_cfg.get("critic_num_layers", 2),
    activation=tqc_cfg.get("activation", "relu"),
    log_std_min=tqc_cfg.get("log_std_min", -20),
    log_std_max=tqc_cfg.get("log_std_max", 2),
    n_critics=tqc_cfg.get("n_critics", 2),
    n_quantiles=tqc_cfg.get("n_quantiles", 25),
    top_quantiles_to_drop=tqc_cfg.get("top_quantiles_to_drop", 2),
    auto_entropy_tuning=tqc_cfg.get("auto_entropy_tuning", True),
    alpha=tqc_cfg.get("alpha", 0.2),
    grad_norm_clip=tqc_cfg.get("grad_norm_clip", 1.0),
    log_dir=None,
    device="cpu",
)

# Load weights
agent.actor.load_state_dict(ckpt["actor"])
for i, c in enumerate(agent.critics):
    c.load_state_dict(ckpt["critics"][i])
for i, ct in enumerate(agent.critics_target):
    ct.load_state_dict(ckpt["critics_target"][i])
if "log_alpha" in ckpt:
    agent.log_alpha.data = ckpt["log_alpha"]
if "total_steps" in ckpt:
    agent._total_steps = ckpt["total_steps"]

print("TQC agent loaded.")

In [ ]:
obs, _ = env.reset()
done = False
equity_curve = [env.initial_balance]   # track equity, not cumulative returns

while not done:
    action = agent.act(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    equity_curve.append(info.get('equity', env.initial_balance))
    done = terminated or truncated

equity_curve = np.array(equity_curve)
cum_return = (equity_curve - equity_curve[0]) / equity_curve[0]   # for plotting

print(f"Final equity: {equity_curve[-1]:.2f}")
print(f"Final cumulative return: {cum_return[-1]:.4f} ({cum_return[-1]*100:.2f}%)")

In [ ]:
# Daily returns (percentage change of equity)
daily_returns = np.diff(equity_curve) / equity_curve[:-1]
total_ret = cum_return[-1]
n = len(daily_returns)
ann_ret = (1 + total_ret) ** (252 / n) - 1 if n > 0 else 0.0
ann_vol = np.std(daily_returns) * np.sqrt(252)
sharpe = ann_ret / ann_vol if ann_vol != 0 else 0.0

# Drawdown (based on equity, not returns)
peak = np.maximum.accumulate(equity_curve)
drawdown = (peak - equity_curve) / peak
max_dd = np.max(drawdown)

win_rate = np.mean(daily_returns > 0)

print("\n--- TQC Performance ---")
print(f"Total Return:       {total_ret:.2%}")
print(f"Annualized Return:  {ann_ret:.2%}")
print(f"Annualized Vol:     {ann_vol:.2%}")
print(f"Sharpe Ratio:       {sharpe:.2f}")
print(f"Max Drawdown:       {max_dd:.2%}")
print(f"Win Rate:           {win_rate:.2%}")

In [ ]:
# Equity curve (cumulative return)
plt.figure(figsize=(14,6))
plt.plot(cum_return, label='TQC', linewidth=2)
plt.title("TQC Equity Curve (Cumulative Return)")
plt.xlabel("Step")
plt.ylabel("Cumulative Return")
plt.grid(True)
plt.legend()
plt.show()

# Drawdown
plt.figure(figsize=(14,3))
plt.fill_between(range(len(drawdown)), 0, drawdown, color='red', alpha=0.4)
plt.plot(drawdown, color='darkred', linewidth=0.5)
plt.title("Drawdown")
plt.xlabel("Step")
plt.ylabel("Drawdown")
plt.grid(True)
plt.show()